# Literature validation — global |∇b| per Bodner et al. (2025)

Standalone reproduction of the reference's global surface buoyancy
gradient magnitude figure, computed with THEIR EXACT OPERATOR on our
snapshot (2012-11-09 12:00), plus the metric-corrected version.

**Reference operator** (from the published plotting code,
`github.com/abodner/submeso_param_net/notebooks/plot_global_grad_b.ipynb`):

1. coarsen surface Θ and S to 1/4°: `coarsen(i=12, j=12,
   boundary="trim").mean()`
2. σ₀ = JMD95(S̄, Θ̄, p=0) − 1000;  b = −g·σ₀/ρ₀ (ρ₀ = 1000)
3. bₓ = b.diff('i')/dxF,  b_y = b.diff('j')/dyF
4. |∇b| = √(bₓ² + b_y²), plotted 0–1.1×10⁻⁶ s⁻²

**Metric issue identified:** after `coarsen(...).mean()`, `diff`
spans 12 native cells (~26 km) but `dxF`/`dyF` are the block-MEAN of
the native spacings (~2.2 km) — `xarray.coarsen.mean()` averages
coordinates, it does not sum them.  The reference's |∇b| is
therefore inflated by a factor of ~12.  This notebook computes both
versions: their operator verbatim (should visually match the
reference figure) and the corrected metric (÷12; a physically
consistent 1/4° gradient, sitting BELOW our native-resolution |∇b|
as coarsening requires).

Bodner, A., Balwada, D., & Zanna, L. (2025). A data-driven approach
for parameterizing ocean submesoscale buoyancy fluxes. J. Adv.
Model. Earth Syst., 17, e2025MS004991.
https://doi.org/10.1029/2025MS004991

No product store is used here — raw Θ/S only (no RUN cell).  Our
native-resolution |∇b| validation lives in
`frontal_structure.ipynb` (§6.2 + consistency checks).

## 1 — Load native-resolution surface Θ and S (rect grid)

In [ ]:
# Raw Theta/Salt from OSN, stitched to the rect grid (same loaders
# as the pipeline).
import numpy as np
import matplotlib.pyplot as plt

import dbof.io.filesystems as filesystems
import dbof.global_dataset_creation.zarr_grid_global as zarr_grid
from dbof.cli.generate_global import load_snapshot
from dbof.global_dataset_creation.data_sources import get_data_source
from dbof.global_dataset_creation.grid_setup import set_up_grid
from dbof.utils.faces_to_latlon import stitch_and_mask

PIPELINE = "SURF"
DATE = "2012-11-09 12:00:00"
S3_ENDPOINT = "https://s3-west.nrp-nautilus.io"

fs_grid, _ = filesystems.create_s3_filesystems(S3_ENDPOINT)
grid_reader = zarr_grid.GlobalGridZarrReader(
    bucket="dbof", folder="LLC4320_GRID_2D",
    dataset_name="llc4320_grid.zarr", fs=fs_grid,
)
XC, YC = grid_reader.lon, grid_reader.lat

ds_grid, land_mask, xgrid = set_up_grid(PIPELINE, None)
ds_raw, ds_merge, it = load_snapshot(
    PIPELINE, DATE, ds_grid, ["Theta", "Salt"],
    surface_only=False, data_source=get_data_source(PIPELINE),
)
print(f"OSN iteration {it}")
print(f"raw dtypes: Theta {ds_merge.Theta.dtype}, "
      f"Salt {ds_merge.Salt.dtype}")

ds_conv = ds_raw.assign({"Theta": ds_merge["Theta"],
                         "Salt": ds_merge["Salt"]})[["Theta", "Salt"]]
mask = {"_land_mask": (ds_merge.hFacC == 0)}
chw = stitch_and_mask(ds_conv, ["Theta", "Salt"], mask)
theta_full, salt_full = chw[0], chw[1]
del chw
print(f"stitched: Theta/Salt {theta_full.shape}")

## 2 — Bodner operator: coarsen Θ/S → σ₀ → b → diff/dxF

Both metric variants are computed.  The 'reference metric' divides
the coarse-cell difference by the block-mean NATIVE spacing (their
`dxF` after `coarsen.mean()`); the 'corrected metric' divides by the
true distance between adjacent 1/4° cell centres (≈12× larger).

In [ ]:
# Bodner et al. (2025) operator, both metric variants.
from dbof.preprocessing.physical_constants import G, RHO0_REFERENCE
import dbof.utils.jmd95_xgcm_implementation as jmd95

R_EARTH = 6371.0e3   # m
BLK = 12             # 12 x 12 -> ~1/4 degree


def _coarsen12(a):
    """12x12 block nanmean, boundary='trim' (xarray-equivalent).

    Inputs: a (2D np.ndarray, full rect grid).
    Outputs: coarsened 2D array.  Generated by LH and Claude
    """
    hb = (a.shape[0] // BLK) * BLK
    wb = (a.shape[1] // BLK) * BLK
    return np.nanmean(
        a[:hb, :wb].reshape(hb // BLK, BLK, wb // BLK, BLK),
        axis=(1, 3))


theta_c = _coarsen12(theta_full)
salt_c = _coarsen12(salt_full)
xc_c = _coarsen12(XC)
yc_c = _coarsen12(YC)
del theta_full, salt_full   # free ~2 GB

# sigma0 and buoyancy from the COARSENED fields (their order),
# using the same JMD95 as the pipeline.
sigma0_c = jmd95.jmd95(salt_c, theta_c, 0.0) - RHO0_REFERENCE
b_c = -G * sigma0_c / RHO0_REFERENCE      # their sign convention

# True spacing between adjacent 1/4-deg cell centres (spherical),
# with the lon difference wrapped across the +/-180 seam.
_dlon = (np.diff(xc_c, axis=1) + 180.0) % 360.0 - 180.0
_coslat = np.cos(np.radians(0.5 * (yc_c[:, 1:] + yc_c[:, :-1])))
dx_true = R_EARTH * np.radians(np.abs(_dlon)) * _coslat
dy_true = R_EARTH * np.radians(np.abs(np.diff(yc_c, axis=0)))

# The reference's denominator: block-MEAN of native spacings
# (= true coarse spacing / 12, since means average rather than sum).
dx_ref = dx_true / BLK
dy_ref = dy_true / BLK

# Gradients (one-sided diff, as in the reference), both metrics.
db_i = np.diff(b_c, axis=1)
db_j = np.diff(b_c, axis=0)


def _gradb(dx, dy):
    """|grad b| from staggered diffs, trimmed to common shape.

    Inputs: dx (J, I-1), dy (J-1, I) denominators [m].
    Outputs: (J-1, I-1) gradient magnitude [s-2].
    Generated by LH and Claude
    """
    bx = db_i / dx
    by = db_j / dy
    return np.sqrt(bx[:-1, :] ** 2 + by[:, :-1] ** 2)


gradb_ref_metric = _gradb(dx_ref, dy_ref)     # reproduces reference
gradb_corrected = _gradb(dx_true, dy_true)    # physically consistent
xc_t, yc_t = xc_c[:-1, :-1], yc_c[:-1, :-1]

for name, a in [("reference metric (dx/12)", gradb_ref_metric),
                ("corrected metric        ", gradb_corrected)]:
    v = a[np.isfinite(a)]
    print(f"{name}: p50={np.percentile(v, 50):.2e}  "
          f"p90={np.percentile(v, 90):.2e}  "
          f"p99={np.percentile(v, 99):.2e}  [s-2]")
print(f"ratio (should be ~{BLK}): "
      f"{np.nanmedian(gradb_ref_metric / gradb_corrected):.2f}")

## 3 — Comparison figure

Top: their operator verbatim (reference metric) on the reference's
0–1.1×10⁻⁶ scale — should visually match their figure.  Bottom: the
corrected metric on a ÷12 scale (0–0.9×10⁻⁷) — same pattern, the
physically consistent amplitude.

In [ ]:
# Two-panel our-data column vs the reference figure.
import cartopy.crs as ccrs

from dbof.plotting.literature_comparison import stacked_side_by_side
from dbof.plotting.pipeline_grids import mask_wrap_cells

LIT_DIR = "../literature_figures/"
PNG = (LIT_DIR + "frontal-structure_gradb_global_LLC4320_"
       "Bodner-et-al(2025).png")

_ref = mask_wrap_cells(xc_t, yc_t, gradb_ref_metric)
_cor = mask_wrap_cells(xc_t, yc_t, gradb_corrected)


def _panel(arr, vmax, label):
    """Global magma panel on a pinned scale.

    Inputs: arr (2D); vmax (float); label (str).
    Outputs: callable(ax).  Generated by LH and Claude
    """
    def _render(ax):
        ax.set_facecolor("black")
        im = ax.pcolormesh(xc_t, yc_t, arr,
                           transform=ccrs.PlateCarree(),
                           cmap="magma", vmin=0.0, vmax=vmax,
                           shading="nearest")
        ax.coastlines(linewidth=0.3, color="gray")
        plt.colorbar(im, ax=ax, orientation="horizontal",
                     fraction=0.04, pad=0.04, extend="max",
                     label=label)
    return _render


stacked_side_by_side(
    [_panel(_ref, 1.1e-6,
            "|\u2207b| (s\u207b\u00b2), reference metric "
            "[0\u20131.1e-6]"),
     _panel(_cor, 1.1e-6 / 12,
            "|\u2207b| (s\u207b\u00b2), corrected metric "
            "[0\u20130.9e-7]")],
    PNG,
    projection=ccrs.Robinson(),
    our_titles=["Bodner operator, reference metric (dx/12)",
                "Bodner operator, corrected metric"],
    caption=("Top-left: the reference's exact operator (coarsen "
             "\u0398/S to 1/4\u00b0, then b, then diff/dxF with "
             "dxF = block-mean native spacing) \u2014 reproduces "
             "the reference figure from OUR snapshot.  Bottom-left: "
             "identical except the gradient uses the true 1/4\u00b0 "
             "cell spacing (\u00d712 larger denominator) \u2014 "
             "the physically consistent coarse |\u2207b|, which "
             "sits below our native-resolution values as coarsening "
             "requires.  Same pattern in all panels; only the "
             "amplitude convention differs.\n"
             "Bodner, A., Balwada, D., & Zanna, L. (2025). A "
             "data-driven approach for parameterizing ocean "
             "submesoscale buoyancy fluxes. J. Adv. Model. Earth "
             "Syst., 17, e2025MS004991. "
             "https://doi.org/10.1029/2025MS004991."),
)
plt.show()

## Discussion

Our native-resolution |∇b| is validated by the store-vs-live
consistency checks (gradb2 = bₓ² + bᵧ² to float precision) and by
direct agreement with Bachman et al. (2021).  The reference's global
map is reproduced exactly by applying their published operator to
our snapshot, confirming that the ~×12 amplitude difference
originates in the reference's metric (block-mean `dxF` used with a
coarse-grid `diff`) rather than in either model field.  With the
corrected metric, the coarse |∇b| sits below the native-resolution
field, as it must.  The figure is motivational in the reference
(region selection) and this factor does not bear on their scientific
results.  Cross-references: native-resolution |∇b| and its
block-mean coarsening → `frontal_structure.ipynb` §6.2; reference
plotting code → `github.com/abodner/submeso_param_net`.